In [34]:
import pandas as pd

In [35]:
df=pd.read_csv("/Users/ravihw18/ML-Algorithms/13-RNN/Dataset/IMDB Dataset.csv")

In [36]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
df.drop_duplicates(inplace=True)

In [38]:
df.shape

(49582, 2)

## Pre-Processing

### 1. Converting to Lowercase

In [39]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [40]:
import re

def remove_urls(text):
    text =re.sub(r"http\S+", "", text)   # (pattern, replace element, string) 
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [41]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [42]:
def remove_html(text):
    text=re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the StopWords

In [43]:
import nltk # natural language tool kit
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [44]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [45]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words= stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [46]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [47]:
# running -> run
# played -> play

from nltk.stem import PorterStemmer

In [48]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token =ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [49]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [50]:
from sklearn.preprocessing import LabelEncoder

le= LabelEncoder()
df["sentiment"]= le.fit_transform(df["sentiment"])

y=df["sentiment"]

### 8. Vectorization

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X= tf.fit_transform(df["review"])

In [52]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset and DataLoader

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [54]:
X_train.shape

(39665, 5000)

In [55]:
import torch
from torch.utils.data import DataLoader,TensorDataset

In [56]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [59]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader=DataLoader(train_set, shuffle=True, batch_size=64)
test_loader=DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN model

In [65]:
import torch
import torch.nn as nn
import torch.optim as optim

In [67]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        # RNN layer
        self.rnn= nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc= nn.Linear(hidden_size,1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [68]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [69]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.35452356934547424
epoch = 2/10 and loss = 0.3399503231048584
epoch = 3/10 and loss = 0.31657958030700684
epoch = 4/10 and loss = 0.2347630113363266
epoch = 5/10 and loss = 0.24152736365795135
epoch = 6/10 and loss = 0.34259477257728577
epoch = 7/10 and loss = 0.17672663927078247
epoch = 8/10 and loss = 0.4009752869606018
epoch = 9/10 and loss = 0.3095845580101013
epoch = 10/10 and loss = 0.43004655838012695


In [70]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.80215791065847
